In [ ]:
# ============================================================
#  ALGO TRADING SIGNAL ENGINE
#  Indicators: EMA · RSI · MACD · Bollinger Bands · Volume
#  Author  : Algo Trader
#  Usage   : python signal_engine.py
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────
#  CONFIG — change these as needed
# ─────────────────────────────────────────────
TICKER   = "RELIANCE.NS"   # NSE: RELIANCE.NS | BSE: RELIANCE.BO | US: AAPL
PERIOD   = "6mo"           # 1mo 3mo 6mo 1y 2y
INTERVAL = "1d"            # 1d 1wk 1mo


# ─────────────────────────────────────────────
#  1. FETCH OHLCV DATA
# ─────────────────────────────────────────────
def fetch_data(ticker: str, period: str, interval: str) -> pd.DataFrame:
    print(f"\n{'='*55}")
    print(f"  Fetching data → {ticker}  |  {period}  |  {interval}")
    print(f"{'='*55}")
    df = yf.download(ticker, period=period, interval=interval, progress=False)
    if df.empty:
        raise ValueError(f"No data returned for {ticker}. Check the ticker symbol.")
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    df.dropna(inplace=True)
    print(f"  Rows loaded : {len(df)}  |  From {df.index[0].date()} to {df.index[-1].date()}")
    return df


# ─────────────────────────────────────────────
#  2. INDICATORS
# ─────────────────────────────────────────────
def add_ema(df: pd.DataFrame, fast=20, slow=50) -> pd.DataFrame:
    df[f'EMA_{fast}'] = df['Close'].ewm(span=fast, adjust=False).mean()
    df[f'EMA_{slow}'] = df['Close'].ewm(span=slow, adjust=False).mean()
    return df


def add_rsi(df: pd.DataFrame, period=14) -> pd.DataFrame:
    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    return df


def add_macd(df: pd.DataFrame, fast=12, slow=26, signal=9) -> pd.DataFrame:
    ema_fast        = df['Close'].ewm(span=fast, adjust=False).mean()
    ema_slow        = df['Close'].ewm(span=slow, adjust=False).mean()
    df['MACD']      = ema_fast - ema_slow
    df['MACD_sig']  = df['MACD'].ewm(span=signal, adjust=False).mean()
    df['MACD_hist'] = df['MACD'] - df['MACD_sig']
    return df


def add_bollinger(df: pd.DataFrame, period=20, std_dev=2) -> pd.DataFrame:
    sma             = df['Close'].rolling(period).mean()
    std             = df['Close'].rolling(period).std()
    df['BB_upper']  = sma + std_dev * std
    df['BB_lower']  = sma - std_dev * std
    df['BB_mid']    = sma
    df['BB_%B']     = (df['Close'] - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'])
    return df


def add_volume_signal(df: pd.DataFrame, period=20) -> pd.DataFrame:
    df['Vol_avg']   = df['Volume'].rolling(period).mean()
    df['Vol_ratio'] = df['Volume'] / df['Vol_avg']
    return df


def compute_all_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = add_ema(df)
    df = add_rsi(df)
    df = add_macd(df)
    df = add_bollinger(df)
    df = add_volume_signal(df)
    df.dropna(inplace=True)
    return df


# ─────────────────────────────────────────────
#  3. SIGNAL LOGIC (confluence model)
# ─────────────────────────────────────────────
def generate_signal(df: pd.DataFrame) -> dict:
    row = df.iloc[-1]   # latest candle

    signals   = {}
    score     = 0       # +1 bullish  |  -1 bearish  per indicator
    reasons   = []

    # --- EMA Trend ---
    if row['EMA_20'] > row['EMA_50']:
        signals['EMA'] = 'BULLISH'
        score += 1
        reasons.append(f"EMA20 ({row['EMA_20']:.2f}) > EMA50 ({row['EMA_50']:.2f}) → uptrend")
    else:
        signals['EMA'] = 'BEARISH'
        score -= 1
        reasons.append(f"EMA20 ({row['EMA_20']:.2f}) < EMA50 ({row['EMA_50']:.2f}) → downtrend")

    # --- RSI Momentum ---
    rsi = row['RSI']
    if rsi < 30:
        signals['RSI'] = 'OVERSOLD (strong buy)'
        score += 2
        reasons.append(f"RSI {rsi:.1f} — oversold zone, high reversal probability")
    elif rsi > 70:
        signals['RSI'] = 'OVERBOUGHT (strong sell)'
        score -= 2
        reasons.append(f"RSI {rsi:.1f} — overbought zone, pullback likely")
    elif 40 <= rsi <= 60:
        signals['RSI'] = 'NEUTRAL'
        reasons.append(f"RSI {rsi:.1f} — neutral zone")
    elif rsi > 60:
        signals['RSI'] = 'BULLISH'
        score += 1
        reasons.append(f"RSI {rsi:.1f} — bullish momentum, not yet overbought")
    else:
        signals['RSI'] = 'BEARISH'
        score -= 1
        reasons.append(f"RSI {rsi:.1f} — bearish momentum, not yet oversold")

    # --- MACD ---
    if row['MACD'] > row['MACD_sig']:
        signals['MACD'] = 'BULLISH'
        score += 1
        reasons.append(f"MACD ({row['MACD']:.3f}) > Signal ({row['MACD_sig']:.3f}) → bullish crossover")
    else:
        signals['MACD'] = 'BEARISH'
        score -= 1
        reasons.append(f"MACD ({row['MACD']:.3f}) < Signal ({row['MACD_sig']:.3f}) → bearish crossover")

    # --- Bollinger Bands ---
    pct_b = row['BB_%B']
    if pct_b < 0.2:
        signals['BB'] = 'OVERSOLD'
        score += 1
        reasons.append(f"Price near BB lower band (%B={pct_b:.2f}) → potential bounce")
    elif pct_b > 0.8:
        signals['BB'] = 'OVERBOUGHT'
        score -= 1
        reasons.append(f"Price near BB upper band (%B={pct_b:.2f}) → potential reversal")
    else:
        signals['BB'] = 'NEUTRAL'
        reasons.append(f"Price in BB mid-zone (%B={pct_b:.2f})")

    # --- Volume Confirmation ---
    vol_ratio = row['Vol_ratio']
    if vol_ratio >= 1.5:
        signals['VOLUME'] = 'HIGH (strong conviction)'
        score += 1
        reasons.append(f"Volume {vol_ratio:.1f}x avg — strong participation confirms move")
    elif vol_ratio < 0.8:
        signals['VOLUME'] = 'LOW (weak conviction)'
        score -= 1
        reasons.append(f"Volume {vol_ratio:.1f}x avg — low participation, signal is weak")
    else:
        signals['VOLUME'] = 'NORMAL'
        reasons.append(f"Volume {vol_ratio:.1f}x avg — normal participation")

    # --- Final Signal ---
    max_score   = 6   # max possible bullish score
    confidence  = round(abs(score) / max_score * 100, 1)

    if score >= 3:
        final = "BUY"
    elif score <= -3:
        final = "SELL"
    else:
        final = "HOLD"

    return {
        "ticker"     : TICKER,
        "signal"     : final,
        "score"      : score,
        "confidence" : confidence,
        "signals"    : signals,
        "reasons"    : reasons,
        "last_candle": row,
    }


# ─────────────────────────────────────────────
#  4. REPORT PRINTER
# ─────────────────────────────────────────────
def print_report(result: dict):
    row = result['last_candle']
    sig = result['signal']

    color = {
        "BUY" : "\033[92m",   # green
        "SELL": "\033[91m",   # red
        "HOLD": "\033[93m",   # yellow
    }
    RESET = "\033[0m"
    BOLD  = "\033[1m"

    print(f"\n{'─'*55}")
    print(f"  LAST CANDLE  ({row.name.date()})")
    print(f"{'─'*55}")
    print(f"  Open   : ₹{row['Open']:.2f}")
    print(f"  High   : ₹{row['High']:.2f}")
    print(f"  Low    : ₹{row['Low']:.2f}")
    print(f"  Close  : ₹{row['Close']:.2f}")
    print(f"  Volume : {int(row['Volume']):,}")

    print(f"\n{'─'*55}")
    print(f"  INDICATORS")
    print(f"{'─'*55}")
    print(f"  EMA 20      : {row['EMA_20']:.2f}   EMA 50  : {row['EMA_50']:.2f}")
    print(f"  RSI 14      : {row['RSI']:.2f}")
    print(f"  MACD        : {row['MACD']:.4f}  Signal : {row['MACD_sig']:.4f}  Hist : {row['MACD_hist']:.4f}")
    print(f"  BB Upper    : {row['BB_upper']:.2f}  Mid : {row['BB_mid']:.2f}  Lower : {row['BB_lower']:.2f}")
    print(f"  BB %B       : {row['BB_%B']:.3f}   (0=lower band  1=upper band)")
    print(f"  Volume Ratio: {row['Vol_ratio']:.2f}x  (vs 20-day avg)")

    print(f"\n{'─'*55}")
    print(f"  SIGNAL BREAKDOWN")
    print(f"{'─'*55}")
    for ind, sig_val in result['signals'].items():
        print(f"  {ind:<8}: {sig_val}")

    print(f"\n{'─'*55}")
    print(f"  REASONS")
    print(f"{'─'*55}")
    for i, r in enumerate(result['reasons'], 1):
        print(f"  {i}. {r}")

    print(f"\n{'='*55}")
    c = color.get(result['signal'], "")
    print(f"  {BOLD}FINAL SIGNAL : {c}{result['signal']}{RESET}{BOLD}   "
          f"(Score: {result['score']:+d}  |  Confidence: {result['confidence']}%){RESET}")
    print(f"{'='*55}\n")

    print("  ⚠️  DISCLAIMER: This is an educational signal engine.")
    print("      Always use stop-loss. Never risk more than 1-2% per trade.\n")


# ─────────────────────────────────────────────
#  5. MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    df     = fetch_data(TICKER, PERIOD, INTERVAL)
    df     = compute_all_indicators(df)
    result = generate_signal(df)
    print_report(result)

In [ ]:
# ============================================================
#  FOR INTRADAY TRADING
#  ALGO TRADING SIGNAL ENGINE
#  Indicators: EMA · RSI · MACD · Bollinger Bands · Volume
#  Author  : Algo Trader
#  Usage   : python signal_engine_intraday.py
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import time
import warnings
from datetime import datetime
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────
#  CONFIG — change these as needed
# ─────────────────────────────────────────────
TICKER        = "BTC-USD"   # NSE: RELIANCE.NS | BSE: RELIANCE.BO | US: AAPL
PERIOD        = "5d"            # keep 5d for intraday
INTERVAL      = "5m"            # 1m 2m 5m 15m 30m 60m
POLL_SECONDS  = 60              # how often to check for new candle (seconds)

# EMA periods (matched to intraday settings)
EMA_FAST = 9
EMA_SLOW = 21


# ─────────────────────────────────────────────
#  1. FETCH OHLCV DATA
# ─────────────────────────────────────────────
def fetch_data(ticker: str, period: str, interval: str) -> pd.DataFrame:
    df = yf.download(ticker, period=period, interval=interval, progress=False)
    if df.empty:
        raise ValueError(f"No data returned for {ticker}. Check the ticker symbol.")
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    df.dropna(inplace=True)
    return df


# ─────────────────────────────────────────────
#  2. INDICATORS
# ─────────────────────────────────────────────
def add_ema(df: pd.DataFrame, fast=EMA_FAST, slow=EMA_SLOW) -> pd.DataFrame:
    df[f'EMA_{fast}'] = df['Close'].ewm(span=fast, adjust=False).mean()
    df[f'EMA_{slow}'] = df['Close'].ewm(span=slow, adjust=False).mean()
    return df


def add_rsi(df: pd.DataFrame, period=7) -> pd.DataFrame:
    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    return df


def add_macd(df: pd.DataFrame, fast=5, slow=13, signal=4) -> pd.DataFrame:
    ema_fast        = df['Close'].ewm(span=fast, adjust=False).mean()
    ema_slow        = df['Close'].ewm(span=slow, adjust=False).mean()
    df['MACD']      = ema_fast - ema_slow
    df['MACD_sig']  = df['MACD'].ewm(span=signal, adjust=False).mean()
    df['MACD_hist'] = df['MACD'] - df['MACD_sig']
    return df


def add_bollinger(df: pd.DataFrame, period=10, std_dev=2) -> pd.DataFrame:
    sma             = df['Close'].rolling(period).mean()
    std             = df['Close'].rolling(period).std()
    df['BB_upper']  = sma + std_dev * std
    df['BB_lower']  = sma - std_dev * std
    df['BB_mid']    = sma
    df['BB_%B']     = (df['Close'] - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'])
    return df


def add_volume_signal(df: pd.DataFrame, period=10) -> pd.DataFrame:
    df['Vol_avg']   = df['Volume'].rolling(period).mean()
    df['Vol_ratio'] = df['Volume'] / df['Vol_avg']
    return df


def compute_all_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = add_ema(df)
    df = add_rsi(df)
    df = add_macd(df)
    df = add_bollinger(df)
    df = add_volume_signal(df)
    df.dropna(inplace=True)
    return df


# ─────────────────────────────────────────────
#  3. SIGNAL LOGIC (confluence model)
# ─────────────────────────────────────────────
def generate_signal(df: pd.DataFrame) -> dict:
    row     = df.iloc[-1]   # latest candle
    signals = {}
    score   = 0             # +1 bullish  |  -1 bearish  per indicator
    reasons = []

    ema_fast_col = f'EMA_{EMA_FAST}'
    ema_slow_col = f'EMA_{EMA_SLOW}'

    # --- EMA Trend ---
    if row[ema_fast_col] > row[ema_slow_col]:
        signals['EMA'] = 'BULLISH'
        score += 1
        reasons.append(
            f"EMA{EMA_FAST} ({row[ema_fast_col]:.2f}) > EMA{EMA_SLOW} ({row[ema_slow_col]:.2f}) → uptrend"
        )
    else:
        signals['EMA'] = 'BEARISH'
        score -= 1
        reasons.append(
            f"EMA{EMA_FAST} ({row[ema_fast_col]:.2f}) < EMA{EMA_SLOW} ({row[ema_slow_col]:.2f}) → downtrend"
        )

    # --- RSI Momentum ---
    rsi = row['RSI']
    if rsi < 30:
        signals['RSI'] = 'OVERSOLD (strong buy)'
        score += 2
        reasons.append(f"RSI {rsi:.1f} — oversold zone, high reversal probability")
    elif rsi > 70:
        signals['RSI'] = 'OVERBOUGHT (strong sell)'
        score -= 2
        reasons.append(f"RSI {rsi:.1f} — overbought zone, pullback likely")
    elif 40 <= rsi <= 60:
        signals['RSI'] = 'NEUTRAL'
        reasons.append(f"RSI {rsi:.1f} — neutral zone")
    elif rsi > 60:
        signals['RSI'] = 'BULLISH'
        score += 1
        reasons.append(f"RSI {rsi:.1f} — bullish momentum, not yet overbought")
    else:
        signals['RSI'] = 'BEARISH'
        score -= 1
        reasons.append(f"RSI {rsi:.1f} — bearish momentum, not yet oversold")

    # --- MACD ---
    if row['MACD'] > row['MACD_sig']:
        signals['MACD'] = 'BULLISH'
        score += 1
        reasons.append(f"MACD ({row['MACD']:.3f}) > Signal ({row['MACD_sig']:.3f}) → bullish crossover")
    else:
        signals['MACD'] = 'BEARISH'
        score -= 1
        reasons.append(f"MACD ({row['MACD']:.3f}) < Signal ({row['MACD_sig']:.3f}) → bearish crossover")

    # --- Bollinger Bands ---
    pct_b = row['BB_%B']
    if pct_b < 0.2:
        signals['BB'] = 'OVERSOLD'
        score += 1
        reasons.append(f"Price near BB lower band (%B={pct_b:.2f}) → potential bounce")
    elif pct_b > 0.8:
        signals['BB'] = 'OVERBOUGHT'
        score -= 1
        reasons.append(f"Price near BB upper band (%B={pct_b:.2f}) → potential reversal")
    else:
        signals['BB'] = 'NEUTRAL'
        reasons.append(f"Price in BB mid-zone (%B={pct_b:.2f})")

    # --- Volume Confirmation ---
    vol_ratio = row['Vol_ratio']
    if vol_ratio >= 1.5:
        signals['VOLUME'] = 'HIGH (strong conviction)'
        score += 1
        reasons.append(f"Volume {vol_ratio:.1f}x avg — strong participation confirms move")
    elif vol_ratio < 0.8:
        signals['VOLUME'] = 'LOW (weak conviction)'
        score -= 1
        reasons.append(f"Volume {vol_ratio:.1f}x avg — low participation, signal is weak")
    else:
        signals['VOLUME'] = 'NORMAL'
        reasons.append(f"Volume {vol_ratio:.1f}x avg — normal participation")

    # --- Final Signal ---
    max_score  = 6
    confidence = round(abs(score) / max_score * 100, 1)

    if score >= 3:
        final = "BUY"
    elif score <= -3:
        final = "SELL"
    else:
        final = "HOLD"

    return {
        "ticker"     : TICKER,
        "signal"     : final,
        "score"      : score,
        "confidence" : confidence,
        "signals"    : signals,
        "reasons"    : reasons,
        "last_candle": row,
    }


# ─────────────────────────────────────────────
#  4. REPORT PRINTER
# ─────────────────────────────────────────────
def print_report(result: dict):
    row = result['last_candle']

    color = {
        "BUY" : "\033[92m",   # green
        "SELL": "\033[91m",   # red
        "HOLD": "\033[93m",   # yellow
    }
    RESET = "\033[0m"
    BOLD  = "\033[1m"

    ema_fast_col = f'EMA_{EMA_FAST}'
    ema_slow_col = f'EMA_{EMA_SLOW}'

    candle_time = row.name.strftime('%Y-%m-%d %H:%M') if hasattr(row.name, 'strftime') else str(row.name)

    print(f"\n{'─'*55}")
    print(f"  LAST CANDLE  ({candle_time})")
    print(f"{'─'*55}")
    print(f"  Open   : ₹{row['Open']:.2f}")
    print(f"  High   : ₹{row['High']:.2f}")
    print(f"  Low    : ₹{row['Low']:.2f}")
    print(f"  Close  : ₹{row['Close']:.2f}")
    print(f"  Volume : {int(row['Volume']):,}")

    print(f"\n{'─'*55}")
    print(f"  INDICATORS")
    print(f"{'─'*55}")
    print(f"  EMA {EMA_FAST:<4}     : {row[ema_fast_col]:.2f}   EMA {EMA_SLOW} : {row[ema_slow_col]:.2f}")
    print(f"  RSI 7       : {row['RSI']:.2f}")
    print(f"  MACD        : {row['MACD']:.4f}  Signal : {row['MACD_sig']:.4f}  Hist : {row['MACD_hist']:.4f}")
    print(f"  BB Upper    : {row['BB_upper']:.2f}  Mid : {row['BB_mid']:.2f}  Lower : {row['BB_lower']:.2f}")
    print(f"  BB %B       : {row['BB_%B']:.3f}   (0=lower band  1=upper band)")
    print(f"  Volume Ratio: {row['Vol_ratio']:.2f}x  (vs 10-bar avg)")

    print(f"\n{'─'*55}")
    print(f"  SIGNAL BREAKDOWN")
    print(f"{'─'*55}")
    for ind, sig_val in result['signals'].items():
        print(f"  {ind:<8}: {sig_val}")

    print(f"\n{'─'*55}")
    print(f"  REASONS")
    print(f"{'─'*55}")
    for i, r in enumerate(result['reasons'], 1):
        print(f"  {i}. {r}")

    print(f"\n{'='*55}")
    c = color.get(result['signal'], "")
    print(f"  {BOLD}FINAL SIGNAL : {c}{result['signal']}{RESET}{BOLD}   "
          f"(Score: {result['score']:+d}  |  Confidence: {result['confidence']}%){RESET}")
    print(f"{'='*55}\n")
    print("  ⚠️  DISCLAIMER: This is an educational signal engine.")
    print("      Always use stop-loss. Never risk more than 1-2% per trade.\n")


# ─────────────────────────────────────────────
#  5. LIVE POLLING LOOP
#     Checks every POLL_SECONDS for a new candle.
#     Only runs the signal engine when a brand-new
#     candle timestamp appears.
# ─────────────────────────────────────────────
def run_live():
    print(f"\n{'='*55}")
    print(f"  LIVE SIGNAL ENGINE STARTED")
    print(f"  Ticker   : {TICKER}")
    print(f"  Interval : {INTERVAL}  |  Period : {PERIOD}")
    print(f"  Polling  : every {POLL_SECONDS}s")
    print(f"  Press Ctrl+C to stop.")
    print(f"{'='*55}")

    last_candle_time = None   # tracks the timestamp of the last processed candle
    iteration        = 0

    while True:
        iteration += 1
        now = datetime.now().strftime('%H:%M:%S')

        try:
            df = fetch_data(TICKER, PERIOD, INTERVAL)
            df = compute_all_indicators(df)

            latest_time = df.index[-1]   # timestamp of the newest candle

            if last_candle_time is None:
                # ── First run: always process ──
                print(f"\n  [{now}]  🚀 Initial signal generated (candle: {latest_time})")
                result          = generate_signal(df)
                last_candle_time = latest_time
                print_report(result)

            elif latest_time != last_candle_time:
                # ── New candle detected ──
                print(f"\n  [{now}]  🕯️  NEW CANDLE detected → {latest_time}")
                result          = generate_signal(df)
                last_candle_time = latest_time
                print_report(result)

            else:
                # ── Same candle, skip ──
                print(f"  [{now}]  ⏳  No new candle yet (last: {latest_time}) — waiting {POLL_SECONDS}s...")

        except Exception as e:
            print(f"  [{now}]  ❌  Error: {e} — retrying in {POLL_SECONDS}s...")

        time.sleep(POLL_SECONDS)


# ─────────────────────────────────────────────
#  6. MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    try:
        run_live()
    except KeyboardInterrupt:
        print("\n\n  ✅  Signal engine stopped by user. Goodbye!\n")